In [1]:
import os
os.environ['OMP_NUM_THREADS'] = '1'   #HGBR runs ~15x faster single-threaded on our small data (macOS)

import sys, warnings
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.discrete.conditional_models import ConditionalLogit
from sklearn.ensemble import HistGradientBoostingRegressor
import shap
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

ROOT = Path.cwd()
while not (ROOT / ".git").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from config import START_YEAR, END_YEAR, HOLDOUT_START, MIN_GAMES_SHARE, MIN_MP
import utils

FEATURES = ['PTS', 'W%', 'FT', 'eFG%', 'DRB', 'AST', 'BLK', 'STL', 'GS', '3P']   #candidates_10
ZCOLS = [c + '_z' for c in FEATURES]

OUT = ROOT / 'results' / 'final'
OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
raw = utils.player_stat_dataframe(START_YEAR, END_YEAR)

def check_mvp_flag(df):
    """Does the MVP flag mark the player with the highest vote share, exactly once per season?"""
    flagged = df['MVP'] == 1
    top_share = df['MVPVS'] == df.groupby('Year')['MVPVS'].transform('max')
    wrong = sorted(df.loc[flagged & ~top_share, 'Year'].unique())
    per_season = flagged.groupby(df['Year']).sum()
    not_one = per_season[per_season != 1].index.tolist()
    print(f'{len(wrong)} of {df["Year"].nunique()} seasons flag the wrong player:', wrong)
    print(f'{len(not_one)} seasons without exactly one MVP flag:', not_one)
    return not wrong and not not_one

#local fix (see binomial_model_nt): the MVP is the player with the highest vote share that season.
#Same fix inside src/utils.py would be: df_tmp['MVP'] = (df_tmp['MVPVS'] == df_tmp['MVPVS'].max()).astype(int)
check_mvp_flag(raw)
raw['MVP'] = (raw['MVPVS'] == raw.groupby('Year')['MVPVS'].transform('max')).astype(int)
assert check_mvp_flag(raw), 'MVP flag is still wrong, stop and check the data'

In [ ]:
def prepare(raw):
    """One row per eligible player-season, with within-season z-scores and MVP_recency."""
    raw = raw.copy()
    #traded players: keep only the season-total row
    is_total = raw['Team'].str.endswith('TM')
    raw['TradedMidSeason'] = is_total.groupby([raw['Year'], raw['Code']]).transform('any').astype(int)
    #season length and games-played share
    season_games = raw[~is_total].groupby('Year')['G'].max()
    raw['SeasonG'] = raw['Year'].map(season_games)
    raw['GP%'] = raw['G'] / raw['SeasonG']
    rows = raw[(raw['TradedMidSeason'] == 0) | is_total]
    #step 1, MVP eligibility: 65 games of 82 with 20+ min (CBA rule), scaled to season length, every season
    min_games = np.ceil(MIN_GAMES_SHARE * rows['SeasonG'])
    eligible = (rows['G'] >= min_games) & (rows['MP'] >= MIN_MP)
    d = rows[eligible].reset_index(drop=True)
    #within-season z-scores
    for col in FEATURES:
        g = d.groupby('Year')[col]
        d[col + '_z'] = (d[col] - g.transform('mean')) / g.transform('std')
    #past-MVP recency: only wins from EARLIER seasons count
    wins = defaultdict(list)
    for y, c in zip(raw.loc[raw['MVP'] == 1, 'Year'], raw.loc[raw['MVP'] == 1, 'Code']):
        wins[c].append(y)
    d['MVP_recency'] = [sum(0.5 ** (y - w - 1) for w in wins[c] if w < y) for y, c in zip(d['Year'], d['Code'])]
    return d

full = prepare(raw)

#check: the development part matches binomial_model_nt (6287 rows, 610 vote-getters)
dev = full[full['Year'] < HOLDOUT_START]
print(dev.shape, (dev['MVPVS'] > 0).sum(), "vote-getters")

In [ ]:
#tuned HGBR settings, read from the tuning results written by binomial_model_nt
tuning = pd.read_csv(OUT / 'gbr_tuning.csv').sort_values(['hits', 'top3', 'avg_rank'], ascending=[False, False, True])
best = tuning.iloc[0].drop(['hits', 'top3', 'avg_rank']).dropna()
INT_PARAMS = ['max_iter', 'max_leaf_nodes', 'min_samples_leaf', 'max_depth']
BEST_GBR = {k: (int(v) if k in INT_PARAMS else float(v) if isinstance(v, (int, float)) else v) for k, v in best.items()}

FINAL_MODELS = {
    'Quasi-binomial':    ('quasi',  ZCOLS,    {}),
    'Gradient boosting': ('hgbr',   FEATURES, {'random_state': 12, **BEST_GBR}),
    'Conditional logit': ('clogit', ZCOLS,    {}),
}
print(FINAL_MODELS['Gradient boosting'][2])

In [ ]:
def predict_holdout(full, models, start=HOLDOUT_START):
    """For each season >= start: train on ALL earlier seasons, predict that season.
    models = {label: (kind, columns, settings)}, kind = 'quasi', 'hgbr' or 'clogit'.
    Returns (predictions for seasons >= start, fits) where fits[season][label] is the fitted model."""
    full = full.copy()
    hold_years = sorted(full.loc[full['Year'] >= start, 'Year'].unique())
    fits = {}
    for label in models:
        full[label] = np.nan
    for yr in hold_years:
        train, test = full['Year'] < yr, full['Year'] == yr
        assert full.loc[train, 'Year'].max() < yr          #never trains on the season it predicts
        for label, (kind, cols, settings) in models.items():
            Xtr, Xte = full.loc[train, cols], full.loc[test, cols]
            if kind == 'quasi':
                fit = sm.GLM(full.loc[train, 'MVPVS'], sm.add_constant(Xtr),
                             family=sm.families.Binomial()).fit(scale='X2')
                full.loc[test, label] = fit.predict(sm.add_constant(Xte, has_constant='add'))
            elif kind == 'hgbr':
                fit = HistGradientBoostingRegressor(**settings).fit(Xtr, full.loc[train, 'MVPVS'])
                full.loc[test, label] = fit.predict(Xte)
            elif kind == 'clogit':
                fit = ConditionalLogit(full.loc[train, 'MVP'], Xtr, groups=full.loc[train, 'Year']).fit(
                    disp=False, skip_hessian=True, maxiter=200)
                full.loc[test, label] = Xte @ fit.params
            fits.setdefault(yr, {})[label] = fit           #keep the model, for the per-season breakdowns
    return full[full['Year'] >= start].reset_index(drop=True), fits

In [ ]:
#check: the same pipeline on development seasons must reproduce binomial_model_nt (2001-2018: QB 11, GB 12, clogit 13)
def score(df, col):
    """MVP hits and top-3 over the seasons in df."""
    hits = top3 = 0
    for yr, g in df.groupby('Year'):
        order = g.sort_values(col, ascending=False)['Code'].tolist()
        rank = order.index(g.loc[g['MVP'] == 1, 'Code'].iloc[0]) + 1
        hits += rank == 1
        top3 += rank <= 3
    return hits, top3, df['Year'].nunique()

dev_res, dev_fits = predict_holdout(dev, FINAL_MODELS, start=2001)
for name in list(FINAL_MODELS) + ['PTS']:
    hits, top3, n = score(dev_res, name)
    print(f'{name}: {hits}/{n} hits, top 3 in {top3}/{n}')

In [ ]:
res, fits = predict_holdout(full, FINAL_MODELS)
print("seasons:", sorted(res['Year'].unique()))
print("players per season:", res.groupby('Year').size().to_dict())
print("one MVP per season:", res.groupby('Year')['MVP'].sum().eq(1).all())
print("missing predictions:", res[list(FINAL_MODELS)].isna().sum().to_dict())

## Hold-out results (2019–2025)
Run the cells below once, after the team has agreed the models above are final. Changing models after seeing these results would defeat the purpose of the hold-out.

In [ ]:
SURFACE, INK, INK2, MUTED, GRID, AXIS = '#fcfcfb', '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7'
BLUE, ORANGE, AQUA = '#2a78d6', '#eb6834', '#1baf7a'

def new_chart(figsize):
    fig, ax = plt.subplots(figsize=figsize, dpi=200)
    fig.patch.set_facecolor(SURFACE); ax.set_facecolor(SURFACE)
    for side in ['top', 'right', 'left']:
        ax.spines[side].set_visible(False)
    ax.spines['bottom'].set_color(AXIS)
    ax.tick_params(colors=MUTED, length=0)
    ax.xaxis.grid(True, color=GRID, linewidth=0.8); ax.set_axisbelow(True)
    return fig, ax

def add_titles(fig, main, sub):
    fig.text(0.02, 0.96, main, fontsize=14, fontweight='bold', color=INK, va='top')
    fig.text(0.02, 0.89, sub, fontsize=10.5, color=INK2, va='top')

names = {'PTS': 'Points', 'W%': 'Team win %', 'FT': 'Free throws', 'eFG%': 'Shooting efficiency (eFG%)',
         'DRB': 'Defensive rebounds', 'AST': 'Assists', 'BLK': 'Blocks', 'STL': 'Steals',
         'GS': 'Games started', '3P': '3-pointers'}
def label(f):
    return names.get(f.removesuffix('_z'), f)

#model label -> prediction column (predict_holdout names the columns after the labels; the baseline is raw points)
models = {'Top scorer wins (baseline)': 'PTS', 'Quasi-binomial': 'Quasi-binomial',
          'Gradient boosting': 'Gradient boosting', 'Conditional logit': 'Conditional logit'}
short = {'Quasi-binomial': 'quasi', 'Gradient boosting': 'gb', 'Conditional logit': 'clogit'}

In [ ]:
def picks_table(df):
    """One row per season: the real MVP, and for each model who it picked (✓ if right) and where it ranked the real MVP."""
    rows = []
    for yr, g in df.groupby('Year'):
        mvp = g.loc[g['MVP'] == 1, 'Code'].iloc[0]
        row = {'Season': f"{yr}–{str(yr + 1)[-2:]}", 'Real MVP': g.loc[g['MVP'] == 1, 'Player'].iloc[0]}
        for name, col in models.items():
            order = g.sort_values(col, ascending=False)
            rank = order['Code'].tolist().index(mvp) + 1
            row[name] = '✓' if rank == 1 else f"{order['Player'].iloc[0]} (MVP ranked {rank})"
        rows.append(row)
    return pd.DataFrame(rows).set_index('Season')

picks = picks_table(res)
picks.to_csv(OUT / 'holdout_picks.csv')
picks

In [ ]:
def plot_model_comparison(df, main, sub, fname):
    rows = [(name, *score(df, col)) for name, col in models.items()]          #(name, hits, top3, n)
    fig, ax = new_chart((8, 3.4))
    bars = ax.barh([r[0] for r in rows], [r[1] for r in rows], height=0.42,
                   color=[MUTED if 'baseline' in r[0] else BLUE for r in rows])
    for b, (name, hits, top3, n) in zip(bars, rows):
        t = ax.text(hits + n * 0.017, b.get_y() + b.get_height() / 2, f'{hits}/{n}',
                    va='center', color=INK, fontsize=11, fontweight='bold')
        ax.annotate(f'  top 3 in {top3}/{n}', xy=(1, 0.5), xycoords=t, va='center', color=INK2, fontsize=9.5)
    n = rows[0][3]
    ax.set_xlim(0, n * 1.25); ax.set_xticks(range(0, n + 1, max(1, n // 6)))
    ax.tick_params(axis='y', colors=INK, labelsize=11)
    ax.set_xlabel("MVPs picked correctly  (top 3 = real MVP among the model's top three picks)", color=INK2, fontsize=9.5)
    add_titles(fig, main, sub)
    fig.tight_layout(rect=(0, 0, 1, 0.83))
    fig.savefig(OUT / fname, facecolor=SURFACE)
    plt.show()

def plot_dev_vs_holdout(dev_df, hold_df, main, sub, fname):
    """Share of seasons each model picked correctly: development seasons (gray) vs hold-out (blue)."""
    names_ = list(models)
    fig, ax = new_chart((8, 3.8))
    y = np.arange(len(names_))
    for offset, df, color, tag in [(0.2, dev_df, AXIS, 'development'), (-0.2, hold_df, BLUE, 'hold-out')]:
        for i, name in enumerate(names_):
            hits, top3, n = score(df, models[name])
            ax.barh(y[i] + offset, hits / n * 100, height=0.36, color=color, label=tag if i == 0 else None)
            ax.text(hits / n * 100 + 1.2, y[i] + offset, f'{hits}/{n}', va='center', fontsize=9.5,
                    color=INK if tag == 'hold-out' else INK2, fontweight='bold' if tag == 'hold-out' else 'normal')
    ax.set_yticks(y); ax.set_yticklabels(names_)
    ax.tick_params(axis='y', colors=INK, labelsize=10.5)
    ax.set_xlim(0, 112); ax.set_xticks(range(0, 101, 20))
    ax.xaxis.set_major_formatter(lambda v, _: f'{v:.0f}%')
    ax.set_xlabel('Share of seasons where the model picked the real MVP', color=INK2, fontsize=9.5)
    ax.legend(loc='lower right', frameon=False, fontsize=9.5, labelcolor=INK2)
    add_titles(fig, main, sub)
    fig.tight_layout(rect=(0, 0, 1, 0.83))
    fig.savefig(OUT / fname, facecolor=SURFACE)
    plt.show()

plot_model_comparison(res, 'MVPs picked correctly, by model',
                      'Hold-out seasons 2019–2025, each predicted using only earlier seasons', 'model_comparison_holdout.png')
plot_dev_vs_holdout(dev_res, res, 'Did performance hold up on new seasons?',
                    'Development 2001–2018 (gray) vs hold-out 2019–2025 (blue)', 'dev_vs_holdout.png')

In [ ]:
def top5_error(df, col):
    """Average |predicted - actual| and average (predicted - actual) vote share over each season's actual top 5."""
    top5 = df.sort_values('MVPVS', ascending=False).groupby('Year').head(5)
    err = top5[col] - top5['MVPVS']
    return err.abs().mean(), err.mean()

for name in ['Quasi-binomial', 'Gradient boosting']:
    for tag, df in [('development 2001-2018', dev_res), ('hold-out 2019-2025', res)]:
        abs_err, bias = top5_error(df, models[name])
        print(f'{name}, {tag}: off by {abs_err:.3f} on average, bias {bias:+.3f}')

In [ ]:
def contributions(df, fits, yr, name):
    """One row per player in season yr, one column per stat: that stat's piece of the player's score,
    under the model that made that season's prediction (coefficient x z-score, or SHAP for gradient boosting)."""
    g = df[df['Year'] == yr]
    kind, cols, _ = FINAL_MODELS[name]
    m = fits[yr][name]
    if kind == 'hgbr':
        return pd.DataFrame(shap.TreeExplainer(m).shap_values(g[cols]), index=g.index, columns=FEATURES)
    return pd.DataFrame(g[cols].values * m.params[cols].values, index=g.index, columns=FEATURES)

def plot_case(df, fits, yr, fname=None):
    """One season, three panels. Each compares the real MVP (blue, left) with one other player:
    the model's pick if it was wrong, or its runner-up if it was right. If blue outweighs orange, the model picked the MVP."""
    g = df[df['Year'] == yr]
    mvp = g['MVP'].idxmax()
    mvp_name = g.loc[mvp, 'Player']
    season = f"{yr}–{str(yr + 1)[-2:]}"
    diffs, titles, n_hits = {}, {}, 0
    for name in short:
        ranked = g[models[name]].sort_values(ascending=False).index
        if ranked[0] == mvp:
            other = ranked[1]
            n_hits += 1
            titles[name] = f"{name}: ✓ right\npicked {mvp_name}\nrunner-up: {g.loc[other, 'Player']}"
        else:
            other = ranked[0]
            titles[name] = f"{name}: ✗ wrong\npicked {g.loc[other, 'Player']}"
        c = contributions(df, fits, yr, name)
        diffs[name] = c.loc[other] - c.loc[mvp]
    diffs = pd.DataFrame(diffs)
    order = diffs.abs().div(diffs.abs().max()).sum(axis=1).sort_values().index
    fig, axes = plt.subplots(1, 3, figsize=(11, 4.8), sharey=True, dpi=200)
    fig.patch.set_facecolor(SURFACE)
    for ax, name in zip(axes, short):
        ax.set_facecolor(SURFACE)
        for side in ['top', 'right', 'left', 'bottom']:
            ax.spines[side].set_visible(False)
        ax.tick_params(length=0, colors=INK)
        ax.set_xticks([])
        ax.axvline(0, color=AXIS, linewidth=1.2)
        d = diffs.loc[order, name]
        ax.barh(range(len(d)), d, height=0.6, color=[ORANGE if v > 0 else BLUE for v in d])
        lim = d.abs().max() * 1.15
        ax.set_xlim(-lim, lim)
        ax.set_title(titles[name], fontsize=10, color=INK, loc='left')
    axes[0].set_yticks(range(len(order)))
    axes[0].set_yticklabels([label(f) for f in order])
    add_titles(fig, f"How each model picked the {season} MVP",
               f"Real MVP: {mvp_name} (blue, left). {n_hits} of 3 models picked him")
    fig.text(0.02, 0.02, f"Blue (left) = stat favored {mvp_name}; orange (right) = favored the other player in that panel. "
             "Bar sizes compare within a model only", fontsize=9, color=INK2)
    fig.tight_layout(rect=(0, 0.05, 1, 0.84))
    fig.savefig(OUT / (fname or f'how_picked_{yr}.png'), facecolor=SURFACE)
    plt.show()

#one chart per hold-out season where at least one model missed
for yr in sorted(res['Year'].unique()):
    g = res[res['Year'] == yr]
    if any(g.loc[g[models[n]].idxmax(), 'MVP'] != 1 for n in short):
        plot_case(res, fits, yr)

In [ ]:
#top 10 predicted players per season (ranked by gradient boosting), with actual vote share and every model's prediction
cols = ['Year', 'Player', 'Team', 'MVPVS', 'MVP'] + list(FINAL_MODELS)
res[cols].sort_values(['Year', 'Gradient boosting'], ascending=[True, False]).groupby('Year').head(10) \
         .to_csv(OUT / 'holdout_predictions.csv', index=False)